# Run Video QA Streamlit App on Colab (GPU) via Cloudflare Tunnel

**Before running:** set the runtime to GPU --> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL - open it to use the app.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/sudeeprana8043-svg/Streamlit_project.git"
REPO_DIR = "/content/Streamlit_project"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Configure model files

The app needs the model artifacts (`binary_model.pkl`, `temporal_adapter.pt`, `checkpoint-140/`, label encoders, etc.).
Choose ONE option below by filling in the values.

- **Option A (Hugging Face repo):** set `MODEL_REPO_ID` to a HF model repo that contains the files.
- **Option B (Google Drive):** set `TEMPORAL_ADAPTER_GDRIVE_URL` and place the rest under `MODEL_DIR`.
- **Option C (mount Drive):** mount your Drive and point `MODEL_DIR` at the folder that already has the files.

In [ ]:
import os, shutil

# The .pkl files and checkpoint-140/ ship in the cloned repo under ./model
# temporal_adapter.pt lives in Google Drive, so mount Drive and copy it in.

from google.colab import drive
drive.mount("/content/drive")

SRC_ADAPTER = "/content/drive/MyDrive/models/temporal_adapter.pt"
DST_ADAPTER = os.path.join("model", "temporal_adapter.pt")

os.makedirs("model", exist_ok=True)
if os.path.exists(SRC_ADAPTER):
    shutil.copy(SRC_ADAPTER, DST_ADAPTER)
    print(f"Copied temporal_adapter.pt -> {DST_ADAPTER}")
else:
    raise FileNotFoundError(f"Not found in Drive: {SRC_ADAPTER}")

# Sanity-check all required model files are present
required = [
    "binary_model.pkl", "model_config.pkl",
    "le_weapon.pkl", "le_location.pkl", "le_people.pkl", "le_super.pkl",
    "temporal_adapter.pt",
    "checkpoint-140/adapter_config.json",
    "checkpoint-140/adapter_model.safetensors",
]
missing = [f for f in required if not os.path.exists(os.path.join("model", f))]
print("Missing:" , missing if missing else "none - all model files present")

## 4. Download the Cloudflare tunnel binary

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 5. Launch Streamlit + public tunnel

This starts Streamlit in the background, then opens a Cloudflare tunnel.
Watch the output for a line like `https://something.trycloudflare.com` and open it in a new tab.
Keep this cell running while you use the app.

In [ ]:
import subprocess, time, os, sys

PORT = 8501

# Start Streamlit (headless) in the background
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "streamlit_app.py",
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("Starting Streamlit... (give it ~20s to boot and load models)")
time.sleep(20)
print("Recent Streamlit log:")
!tail -n 20 /content/streamlit.log

# Open the public tunnel (this blocks and prints the trycloudflare URL)
print("\n=== Public URL will appear below (look for *.trycloudflare.com) ===\n")
!./cloudflared tunnel --url http://localhost:$PORT